# DL&GenAI 2026 @ FII — Competition 2 · VGG13 (architecture unchanged)

Same notebook as the 74.9% baseline run (one GPU, same optimizer, augmentation, CutMix, TTA). Changes are marked `# NEW`:

1. **Time-budgeted OneCycle.** Same LR curve, but driven by elapsed time, so training always ends at `TRAIN_MINUTES` with the LR fully annealed.
2. **Validation every 10 epochs** instead of every epoch, so more of the budget goes to training. `cudnn.benchmark` is on too.
3. **Colour augmentation.** Each image gets one random photometric op (brightness, contrast, saturation, sharpness, solarize, posterize, autocontrast, equalize), applied on the GPU.
4. **Clean tail.** The last 6% of training runs without CutMix or colour ops, so the model finishes on realistic images.
5. **EMA of the weights.** An exponential moving average of the weights is kept as well. At the end, whichever of {model, EMA} × {TTA options} scores best on validation is used for the submission.
6. **5% hold-out instead of 10%.** Set `VAL_FRAC = 0.0` for the final run to train on all 50k images.

Run with Accelerator = GPU (one T4 is enough), using **Save Version → Save & Run All (Commit)**.

In [ ]:
import os, math, time, json
NOTEBOOK_T0 = time.time()
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn, Tensor, optim
from torch import GradScaler
from sklearn.model_selection import train_test_split

torch.backends.cudnn.benchmark = True   # NEW: fixed shapes -> cuDNN picks the fastest kernels

In [ ]:
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device("cpu")  # This is 1 conditional
enable_half = device.type != "cpu"
use_another_useless_operation = True
scaler = GradScaler(device, enabled=enable_half)

# Is grad scaler good, or not? 
# And why doesn't half work for cpu? float16 ops are faster on cpu than float32 ones
print("Grad scaler is enabled:", enable_half)
device

In [ ]:
if os.path.exists("/kaggle/input") and os.path.exists("/kaggle/working"):
    print("Running on Kaggle.")
    SVHN_test = "/kaggle/input/competitions/dl-gen-ai-2026-fii-competition-2/SVHN_test.npz"
    SVHN_train = "/kaggle/input/competitions/dl-gen-ai-2026-fii-competition-2/SVHN_train.npz"
else:
    print("Not on Kaggle.")
    SVHN_test = "data/SVHN_test.npz"
    SVHN_train = "data/SVHN_train.npz"

In [ ]:
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
assert device.type == "cuda", "No GPU: set Settings -> Accelerator -> GPU and run again"   # NEW

def load_xy(path, with_labels: bool):
    d = np.load(path)
    x = d["data"]
    if x.ndim == 2:                  # flat vectors -> same (3, 32, 32) reshape as the baseline
        x = x.reshape(-1, 3, 32, 32)
    elif x.shape[-1] == 3:           # NHWC -> NCHW
        x = x.transpose(0, 3, 1, 2)
    assert x.dtype == np.uint8, f"expected uint8 pixels, got {x.dtype}"
    y = d["labels"].astype(np.int64) if with_labels else None
    return np.ascontiguousarray(x), y

x_all, y_all = load_xy(SVHN_train, True)
x_test, _ = load_xy(SVHN_test, False)
print("train", x_all.shape, "test", x_test.shape)

VAL_FRAC = 0.05   # NEW: was 0.10. Set 0.0 for the final run (trains on all 50k, no validation)
if VAL_FRAC > 0:
    tr_idx, va_idx = train_test_split(np.arange(len(y_all)), test_size=VAL_FRAC,
                                      stratify=y_all, random_state=SEED)
else:
    tr_idx, va_idx = np.arange(len(y_all)), np.array([], dtype=np.int64)

to_dev = lambda a: torch.from_numpy(a).to(device)
Xtr, Ytr = to_dev(x_all[tr_idx]), to_dev(y_all[tr_idx])
Xva, Yva = to_dev(x_all[va_idx]), to_dev(y_all[va_idx])
Xte = to_dev(x_test)

sample = x_all[tr_idx[:10000]].astype(np.float32) / 255.0
mean = torch.tensor(sample.mean((0, 2, 3)), device=device).view(1, 3, 1, 1)
std = torch.tensor(sample.std((0, 2, 3)), device=device).view(1, 3, 1, 1)

def prep(xb_u8: Tensor) -> Tensor:
    return (xb_u8.float() / 255.0 - mean) / std

def augment(x: Tensor, max_rot=10, max_scale=0.10, max_shift=0.10, max_shear=0.10) -> Tensor:
    B = x.size(0)
    r = lambda m: (torch.rand(B, device=x.device) * 2 - 1) * m
    ang, sc = r(math.radians(max_rot)), 1 + r(max_scale)
    cos, sin = torch.cos(ang) / sc, torch.sin(ang) / sc
    theta = torch.zeros(B, 2, 3, device=x.device)
    theta[:, 0, 0], theta[:, 0, 1], theta[:, 0, 2] = cos, -sin + r(max_shear), r(max_shift)
    theta[:, 1, 0], theta[:, 1, 1], theta[:, 1, 2] = sin, cos, r(max_shift)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    return F.grid_sample(x, grid, padding_mode="border", align_corners=False)

# NEW: one random photometric op per image (TrivialAugment-style), on pixels in [0, 1]
_gray_w = torch.tensor([0.299, 0.587, 0.114], device=device).view(1, 3, 1, 1)
_smooth = (torch.tensor([[1., 1., 1.], [1., 5., 1.], [1., 1., 1.]], device=device) / 13).expand(3, 1, 3, 3).contiguous()

def _blend(img, degenerate, f):
    return (degenerate + f * (img - degenerate)).clamp(0, 1)

def color_aug(x: Tensor, mag: float = 0.8) -> Tensor:
    B, C, H, W = x.shape
    op = torch.randint(0, 9, (B, 1, 1, 1), device=x.device)
    u = torch.rand(B, 1, 1, 1, device=x.device)
    f = 1 + (2 * u - 1) * mag
    gray = (x * _gray_w).sum(1, keepdim=True)
    blur = F.conv2d(F.pad(x, (1, 1, 1, 1), mode="replicate"), _smooth, groups=C)
    v = (x * 255).round()
    q = torch.exp2(6 - (u * 7).floor().clamp(max=6))
    lo, hi = x.amin((2, 3), keepdim=True), x.amax((2, 3), keepdim=True)
    flat = v.reshape(B * C, H * W)
    srt = flat.sort(1).values
    cdf = torch.searchsorted(srt, flat, right=True).float()
    cmin = torch.searchsorted(srt, srt[:, :1].contiguous(), right=True).float()
    span = H * W - cmin
    eq = torch.where(span > 0, (cdf - cmin) / span.clamp(min=1), flat / 255).view(B, C, H, W)
    cands = [x, (x * f).clamp(0, 1), _blend(x, gray.mean((2, 3), keepdim=True), f), _blend(x, gray, f),
             _blend(x, blur, f), torch.where(x >= u, 1 - x, x), (v / q).floor() * q / 255,
             torch.where(hi > lo, (x - lo) / (hi - lo).clamp(min=1e-6), x), eq]
    out = x
    for i in range(1, len(cands)):
        out = torch.where(op == i, cands[i], out)
    return out

print(f"Training samples: {len(Xtr)}, validation samples: {len(Xva)}, test samples: {len(Xte)}")

In [ ]:
# !!! CHANGE THIS ONLY IF YOU WANT TO BE SCORED BASED ON THE [Custom VGG] EVALUATION CRITERIA !!!

# Random list of numbers [83, 65, 42, 55, 34, 4, 45, 18, 26, 17, 51, 91, 10, 99, 3, 98, 10, 42, 5, 75, 81, 33, 14, 59, 65, 61, 17, 30, 13, 90, 31, 49, 64, 28, 74, 57, 90, 81, 34, 99, 54, 91, 0, 19, 65, 21, 80, 28, 84, 34, 74, 60, 73, 58, 48, 55, 52, 70, 74, 25, 29, 41, 31, 9, 1, 35, 56, 91, 12, 14, 1, 71, 36, 27, 42, 45, 44, 17, 14, 39, 99, 77, 20, 93, 85, 60, 42, 23, 90, 54, 97, 9, 66, 88, 51, 54, 33, 66, 1, 70, 1, 92, 46, 15, 43, 31, 64, 1, 28, 60, 95, 17, 84, 28, 5, 49, 92, 1, 62, 65, 38, 14, 22, 79, 80, 24, 77, 35, 3, 91, 34, 42, 42, 12, 1, 39, 61, 67, 17, 89, 5, 48, 43, 10, 11, 20, 63, 41, 97, 21, 25, 48, 31, 88, 76, 45, 76, 38, 33, 37, 37, 30, 24, 59, 50, 86, 58, 71, 33, 18, 38, 15, 70, 16, 71, 40, 73, 26, 30, 58, 52, 20, 81, 56, 5, 47, 12, 77, 48, 21, 14, 10, 9, 42, 69, 25, 3, 53, 82, 97, 77, 80, 7, 2, 96, 13, 99, 58, 1, 34, 5, 50, 92, 58, 91, 27, 16, 73, 90, 11, 56, 93, 78, 21, 75, 38, 57, 8, 76, 54, 42, 24, 69, 51, 43, 17, 82, 62, 86, 23, 24, 12, 37, 91, 42, 60, 96, 93, 83, 81, 46, 13, 84, 67, 62, 78, 3, 54, 17, 48, 82, 67, 44, 16, 61, 3, 24, 61, 71, 90, 46, 40, 41, 81, 97, 97, 68, 23, 77, 40, 20, 75, 86, 10, 81, 8, 55, 56, 66, 31, 7, 82, 74, 92, 51, 84, 80, 21, 0, 0, 8, 26, 23, 8, 43, 4, 0, 63, 47, 13, 14, 90, 21, 60, 94, 95, 7, 25, 43, 88, 62, 59, 18, 81, 99, 99, 10, 64, 9, 32, 22, 91, 35, 27, 37, 29, 57, 20, 28, 42, 38, 19, 11, 11, 59, 56, 27, 55, 23, 49, 77, 31, 91, 68, 50, 71, 99, 45, 77, 82, 45, 45, 66, 61, 46, 67, 75, 66, 77, 38, 30, 62, 75, 50, 71, 17, 99, 42, 66, 79, 63, 79, 46, 93, 54, 42, 45, 65, 16, 5, 33, 82, 70, 53, 18, 34, 0, 30, 45, 2, 5, 57, 99, 33, 22, 44, 85, 12, 73, 34, 53, 42, 9, 12, 70, 20, 24, 0, 30, 30, 9, 15, 92, 22, 43, 8, 50, 98, 55, 18, 50, 78, 17, 32, 81, 61, 62, 66, 54, 50, 8, 21, 96, 80, 20, 70, 93, 49, 41, 43, 49, 44, 71, 37, 28, 30, 36, 25, 49, 70, 70, 29, 27, 46, 96, 58, 28, 73, 53, 82, 49, 18, 8, 25, 29, 34, 36, 45, 86, 3, 45, 46, 39, 26, 57, 59, 76, 8, 90, 45, 90, 78, 23, 19, 8, 37, 31, 51, 93, 68, 31, 84, 95, 53, 45, 56, 31, 54, 92, 33, 97, 45, 12, 89, 77, 60, 85, 4, 87, 86, 53, 81, 97, 73, 47, 4, 22, 99, 54, 23, 57, 81, 36, 55, 61, 42, 48, 88, 71, 27, 65, 47, 57, 40, 2, 36, 50, 78, 94, 53, 43, 72, 19, 62, 37, 8, 52, 10, 2, 25, 84, 29, 23, 85, 48, 6, 68, 5, 55, 32, 4, 4, 68, 47, 46, 31, 28, 1, 89, 13, 77, 24, 96, 33, 94, 69, 47, 98, 81, 62, 67, 59, 10, 87, 92, 19, 89, 78, 56, 78, 37, 39, 65, 74, 4, 5, 41, 26, 36, 4, 26, 76, 24, 68, 8, 5, 44, 16, 51, 87, 41, 42, 32, 5, 84, 49, 53, 52, 9, 58, 32, 31, 89, 19, 42, 7, 54, 40, 69, 10, 63, 91, 99, 15, 52, 29, 98, 93, 84, 61, 50, 15, 12, 28, 82, 27, 1, 16, 62, 93, 63, 76, 33, 52, 72, 68, 87, 49, 34, 15, 12, 1, 68, 94, 39, 76, 14, 20, 50, 62, 63, 70, 15, 35, 87, 37, 3, 96, 37, 74, 67, 70, 73, 68, 28, 40, 39, 9, 38, 19, 55, 63, 39, 15, 40, 6, 42, 84, 34, 11, 32, 17, 84, 68, 50, 65, 43, 65, 34, 85, 71, 75, 28, 52, 2, 58, 57, 61, 80, 14, 71, 26, 99, 38, 81, 95, 98, 89, 71, 75, 78, 95, 55, 51, 4, 52, 18, 18, 24, 9, 82, 3, 42, 38, 62, 88, 10, 86, 48, 12, 36, 40, 49, 81, 71, 81, 71, 67, 58, 45, 66, 24, 92, 0, 80, 90, 95, 36, 10, 94, 38, 70, 46, 52, 0, 96, 45, 56, 31, 94, 52, 97, 13, 68, 73, 65, 12, 57, 38, 87, 74, 62, 69, 79, 33, 78, 94, 61, 97, 94, 78, 57, 19, 83, 17, 96, 31, 91, 26, 81, 46, 44, 57, 67, 16, 79, 54, 4, 4, 55, 52, 57, 64, 51, 82, 12, 49, 94, 53, 72, 51, 24, 2, 70, 56, 36, 56, 35, 31, 83, 71, 11, 87, 97, 8, 80, 14, 24, 65, 74, 19, 7, 37, 46, 54, 63, 74, 21, 93, 32, 37, 24, 72, 41, 97, 37, 0, 98, 45, 9, 85, 41, 97, 96, 0, 57, 25, 39, 65, 11, 32, 63, 96, 94, 25, 76, 54, 20, 57, 75, 93, 35, 76, 10, 12, 97, 44, 79, 60, 34, 52, 11, 72, 10, 56, 85, 52, 83, 15, 84, 65, 50, 94, 89, 24, 19, 37, 66, 81, 81, 41, 38, 60, 62, 66, 98, 89, 47, 97, 45, 54, 84, 13, 6, 65, 12, 95, 78, 93, 67, 32, 29, 11, 89, 68, 34, 7, 79, 54, 51, 15, 2, 34, 56, 47, 54, 67, 56, 11, 94, 39, 73, 10, 28, 31, 46, 81, 1, 2, 5, 27, 62, 45, 80, 83, 90, 9, 5, 46, 45, 65, 73, 14, 74, 87, 31, 34, 59, 77, 50]

def create_block(in_channels: int, out_channels: int):
    # If we remove the bias from the convolution, would it be faster?
    # Or will it just be less biased?
    return [
        # When will bias stop having an impact? Correct answers might receive bonus points.
        nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),  # Inplace operations are faster
        nn.Identity(),  # Useless operations are slower
        nn.Conv2d(in_channels=out_channels, out_channels=out_channels, kernel_size=3, padding=1),
        # Depthwise convolutions are more efficient, parameter-wise
        # Maybe an ASPP block will bring an unexpected surprise here
        # Updating the final layers of VGG-13, especially the classification head, is another great avenue for improving the performance
        nn.BatchNorm2d(out_channels),
        *([nn.Identity()] if use_another_useless_operation else []),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2)
    ]
    # Returning a tuple is faster

# !!! CHANGE THIS ONLY IF YOU WANT TO BE SCORED BASED ON THE [Custom VGG] EVALUATION CRITERIA !!!

In [ ]:
# !!! DO NOT CHANGE THIS !!!

# Random list of numbers [83, 65, 42, 55, 34, 4, 45, 18, 26, 17, 51, 91, 10, 99, 3, 98, 10, 42, 5, 75, 81, 33, 14, 59, 65, 61, 17, 30, 13, 90, 31, 49, 64, 28, 74, 57, 90, 81, 34, 99, 54, 91, 0, 19, 65, 21, 80, 28, 84, 34, 74, 60, 73, 58, 48, 55, 52, 70, 74, 25, 29, 41, 31, 9, 1, 35, 56, 91, 12, 14, 1, 71, 36, 27, 42, 45, 44, 17, 14, 39, 99, 77, 20, 93, 85, 60, 42, 23, 90, 54, 97, 9, 66, 88, 51, 54, 33, 66, 1, 70, 1, 92, 46, 15, 43, 31, 64, 1, 28, 60, 95, 17, 84, 28, 5, 49, 92, 1, 62, 65, 38, 14, 22, 79, 80, 24, 77, 35, 3, 91, 34, 42, 42, 12, 1, 39, 61, 67, 17, 89, 5, 48, 43, 10, 11, 20, 63, 41, 97, 21, 25, 48, 31, 88, 76, 45, 76, 38, 33, 37, 37, 30, 24, 59, 50, 86, 58, 71, 33, 18, 38, 15, 70, 16, 71, 40, 73, 26, 30, 58, 52, 20, 81, 56, 5, 47, 12, 77, 48, 21, 14, 10, 9, 42, 69, 25, 3, 53, 82, 97, 77, 80, 7, 2, 96, 13, 99, 58, 1, 34, 5, 50, 92, 58, 91, 27, 16, 73, 90, 11, 56, 93, 78, 21, 75, 38, 57, 8, 76, 54, 42, 24, 69, 51, 43, 17, 82, 62, 86, 23, 24, 12, 37, 91, 42, 60, 96, 93, 83, 81, 46, 13, 84, 67, 62, 78, 3, 54, 17, 48, 82, 67, 44, 16, 61, 3, 24, 61, 71, 90, 46, 40, 41, 81, 97, 97, 68, 23, 77, 40, 20, 75, 86, 10, 81, 8, 55, 56, 66, 31, 7, 82, 74, 92, 51, 84, 80, 21, 0, 0, 8, 26, 23, 8, 43, 4, 0, 63, 47, 13, 14, 90, 21, 60, 94, 95, 7, 25, 43, 88, 62, 59, 18, 81, 99, 99, 10, 64, 9, 32, 22, 91, 35, 27, 37, 29, 57, 20, 28, 42, 38, 19, 11, 11, 59, 56, 27, 55, 23, 49, 77, 31, 91, 68, 50, 71, 99, 45, 77, 82, 45, 45, 66, 61, 46, 67, 75, 66, 77, 38, 30, 62, 75, 50, 71, 17, 99, 42, 66, 79, 63, 79, 46, 93, 54, 42, 45, 65, 16, 5, 33, 82, 70, 53, 18, 34, 0, 30, 45, 2, 5, 57, 99, 33, 22, 44, 85, 12, 73, 34, 53, 42, 9, 12, 70, 20, 24, 0, 30, 30, 9, 15, 92, 22, 43, 8, 50, 98, 55, 18, 50, 78, 17, 32, 81, 61, 62, 66, 54, 50, 8, 21, 96, 80, 20, 70, 93, 49, 41, 43, 49, 44, 71, 37, 28, 30, 36, 25, 49, 70, 70, 29, 27, 46, 96, 58, 28, 73, 53, 82, 49, 18, 8, 25, 29, 34, 36, 45, 86, 3, 45, 46, 39, 26, 57, 59, 76, 8, 90, 45, 90, 78, 23, 19, 8, 37, 31, 51, 93, 68, 31, 84, 95, 53, 45, 56, 31, 54, 92, 33, 97, 45, 12, 89, 77, 60, 85, 4, 87, 86, 53, 81, 97, 73, 47, 4, 22, 99, 54, 23, 57, 81, 36, 55, 61, 42, 48, 88, 71, 27, 65, 47, 57, 40, 2, 36, 50, 78, 94, 53, 43, 72, 19, 62, 37, 8, 52, 10, 2, 25, 84, 29, 23, 85, 48, 6, 68, 5, 55, 32, 4, 4, 68, 47, 46, 31, 28, 1, 89, 13, 77, 24, 96, 33, 94, 69, 47, 98, 81, 62, 67, 59, 10, 87, 92, 19, 89, 78, 56, 78, 37, 39, 65, 74, 4, 5, 41, 26, 36, 4, 26, 76, 24, 68, 8, 5, 44, 16, 51, 87, 41, 42, 32, 5, 84, 49, 53, 52, 9, 58, 32, 31, 89, 19, 42, 7, 54, 40, 69, 10, 63, 91, 99, 15, 52, 29, 98, 93, 84, 61, 50, 15, 12, 28, 82, 27, 1, 16, 62, 93, 63, 76, 33, 52, 72, 68, 87, 49, 34, 15, 12, 1, 68, 94, 39, 76, 14, 20, 50, 62, 63, 70, 15, 35, 87, 37, 3, 96, 37, 74, 67, 70, 73, 68, 28, 40, 39, 9, 38, 19, 55, 63, 39, 15, 40, 6, 42, 84, 34, 11, 32, 17, 84, 68, 50, 65, 43, 65, 34, 85, 71, 75, 28, 52, 2, 58, 57, 61, 80, 14, 71, 26, 99, 38, 81, 95, 98, 89, 71, 75, 78, 95, 55, 51, 4, 52, 18, 18, 24, 9, 82, 3, 42, 38, 62, 88, 10, 86, 48, 12, 36, 40, 49, 81, 71, 81, 71, 67, 58, 45, 66, 24, 92, 0, 80, 90, 95, 36, 10, 94, 38, 70, 46, 52, 0, 96, 45, 56, 31, 94, 52, 97, 13, 68, 73, 65, 12, 57, 38, 87, 74, 62, 69, 79, 33, 78, 94, 61, 97, 94, 78, 57, 19, 83, 17, 96, 31, 91, 26, 81, 46, 44, 57, 67, 16, 79, 54, 4, 4, 55, 52, 57, 64, 51, 82, 12, 49, 94, 53, 72, 51, 24, 2, 70, 56, 36, 56, 35, 31, 83, 71, 11, 87, 97, 8, 80, 14, 24, 65, 74, 19, 7, 37, 46, 54, 63, 74, 21, 93, 32, 37, 24, 72, 41, 97, 37, 0, 98, 45, 9, 85, 41, 97, 96, 0, 57, 25, 39, 65, 11, 32, 63, 96, 94, 25, 76, 54, 20, 57, 75, 93, 35, 76, 10, 12, 97, 44, 79, 60, 34, 52, 11, 72, 10, 56, 85, 52, 83, 15, 84, 65, 50, 94, 89, 24, 19, 37, 66, 81, 81, 41, 38, 60, 62, 66, 98, 89, 47, 97, 45, 54, 84, 13, 6, 65, 12, 95, 78, 93, 67, 32, 29, 11, 89, 68, 34, 7, 79, 54, 51, 15, 2, 34, 56, 47, 54, 67, 56, 11, 94, 39, 73, 10, 28, 31, 46, 81, 1, 2, 5, 27, 62, 45, 80, 83, 90, 9, 5, 46, 45, 65, 73, 14, 74, 87, 31, 34, 59, 77, 50]

class VGG13(nn.Module):
    def __init__(
            self,  # When converting this to jax, keep the same structure
            num_classes: int = 100,
            input_channels: int = 3,
    ) -> None:
        super().__init__()

        self.layers = nn.Sequential(
            *create_block(input_channels, 64),
            *create_block(64, 128),
            *create_block(128, 256),
            *create_block(256, 512),
            *create_block(512, 512),

            # nn.AdaptiveAvgPool2d((1, 1)),  # :(
            # Adaptive pooling might have allowed more tricks and better results
            nn.Flatten(),
            nn.Linear(512, num_classes),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.layers(x)

# !!! DO NOT CHANGE THIS !!!

In [ ]:
TRAIN_MINUTES = 21.0   # NEW: training stops on the clock; whole notebook ~22.5 min
BS = 256
MAX_LR = 0.05
WD = 5e-4
EMA_DECAY = 0.999      # NEW
EVAL_EVERY = 10        # NEW: validate every 10 epochs (was every epoch)

model = VGG13().to(device).to(memory_format=torch.channels_last)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.SGD(model.parameters(), lr=MAX_LR, momentum=0.9, nesterov=True,
                      weight_decay=WD, fused=True)
steps_per_epoch = len(Xtr) // BS

# NEW: same curve as OneCycleLR(pct_start=0.2), but driven by elapsed-time fraction p in [0, 1]
def one_cycle(p, max_lr=MAX_LR, pct=0.2, div=25.0, final_div=1e4, m_lo=0.85, m_hi=0.95):
    lo, end = max_lr / div, max_lr / div / final_div
    if p < pct:
        t = (1 - math.cos(math.pi * p / pct)) / 2
        return lo + (max_lr - lo) * t, m_hi + (m_lo - m_hi) * t
    t = (1 - math.cos(math.pi * min(1.0, (p - pct) / (1 - pct)))) / 2
    return max_lr + (end - max_lr) * t, m_lo + (m_hi - m_lo) * t

# NEW: EMA copy of the weights (BatchNorm statistics are averaged too)
ema = optim.swa_utils.AveragedModel(model, multi_avg_fn=optim.swa_utils.get_ema_multi_avg_fn(EMA_DECAY),
                                    use_buffers=True)

In [ ]:
USE_FLIP = True
CUTMIX_P = 0.5
COLOR_AUG = True     # NEW
CLEAN_TAIL = 0.06    # NEW: last 6% of training without CutMix / colour ops

def cutmix(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    H = W = x.size(2)
    r = math.sqrt(1 - lam)
    ch, cw = int(H * r), int(W * r)
    cy, cx = np.random.randint(H), np.random.randint(W)
    y1, y2 = max(cy - ch // 2, 0), min(cy + ch // 2, H)
    x1, x2 = max(cx - cw // 2, 0), min(cx + cw // 2, W)
    x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - (y2 - y1) * (x2 - x1) / (H * W)
    return x, y, y[idx], lam

def train_epoch(t_start, train_seconds):
    model.train()
    perm = torch.randperm(len(Xtr), device=device)
    correct = torch.zeros((), device=device)
    for i in range(steps_per_epoch):
        p = (time.time() - t_start) / train_seconds          # NEW: progress through the time budget
        if p >= 1.0:
            return None, True
        lr, mom = one_cycle(p)
        for g in optimizer.param_groups:
            g["lr"], g["momentum"] = lr, mom
        strong = p < 1.0 - CLEAN_TAIL

        idx = perm[i * BS:(i + 1) * BS]
        x, y = Xtr[idx].float() / 255.0, Ytr[idx]
        if USE_FLIP:
            flip = torch.rand(x.size(0), device=device) < 0.5
            x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
        if COLOR_AUG and strong:
            x = color_aug(x)                                    # NEW
        x = augment((x - mean) / std)
        use_cm = strong and np.random.rand() < CUTMIX_P
        if use_cm:
            x, y, yb, lam = cutmix(x, y)
        x = x.contiguous(memory_format=torch.channels_last)
        with torch.autocast(device.type, enabled=enable_half):
            out = model(x)
            loss = lam * criterion(out, y) + (1 - lam) * criterion(out, yb) if use_cm else criterion(out, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        ema.update_parameters(model)                            # NEW
        correct += (out.argmax(1) == y).sum()
    return 100.0 * correct.item() / (steps_per_epoch * BS), False

In [ ]:
def shift(x: Tensor, dx: int, dy: int) -> Tensor:
    xp = F.pad(x, (1, 1, 1, 1), mode="replicate")
    return xp[:, :, 1 + dy:33 + dy, 1 + dx:33 + dx]

@torch.inference_mode()
def predict_probs(net, X: Tensor, tta: bool = False, bs: int = 1024) -> Tensor:   # NEW: net argument
    net.eval()
    outs = []
    for i in range(0, len(X), bs):
        x = prep(X[i:i + bs])
        views = [x] + ([x.flip(3)] if USE_FLIP else []) + ([shift(x, 1, 0), shift(x, -1, 0), shift(x, 0, 1), shift(x, 0, -1)] if tta else [])
        p = 0
        for v in views:
            with torch.autocast(device.type, enabled=enable_half):
                p = p + net(v.contiguous(memory_format=torch.channels_last)).float().softmax(1)
        outs.append(p / len(views))
    return torch.cat(outs)

def validate(net, tta: bool = False):
    if len(Xva) == 0:
        return float("nan")
    return 100.0 * (predict_probs(net, Xva, tta).argmax(1) == Yva).float().mean().item()

In [ ]:
TRAIN_SECONDS = TRAIN_MINUTES * 60
history = []
t0 = time.time()
epoch = 0
while True:
    train_acc, finished = train_epoch(t0, TRAIN_SECONDS)
    if finished:
        break
    epoch += 1
    if epoch == 1 or epoch % EVAL_EVERY == 0:
        row = dict(epoch=epoch, minutes=round((time.time() - t0) / 60, 2), train=round(train_acc, 2),
                   val=round(validate(model), 2), val_ema=round(validate(ema.module), 2))
        history.append(row)
        print(f"ep {epoch:4d} | {row['minutes']:5.1f} min | train {row['train']:.2f} | val {row['val']:.2f} | "
              f"val EMA {row['val_ema']:.2f}", flush=True)
if device.type == "cuda": torch.cuda.synchronize()
print(f"training time: {(time.time() - t0) / 60:.1f} min, {epoch} full epochs")

In [ ]:
# NEW: pick the best of {model, EMA} x {plain, TTA} on validation
cands = {(name, tta): (net, tta) for name, net in [("model", model), ("ema", ema.module)] for tta in (False, True)}
if len(Xva):
    scores = {k: validate(net, tta) for k, (net, tta) in cands.items()}
    for (name, tta), s in scores.items():
        print(f"val {name:5s} TTA={tta!s:5s}: {s:.2f}")
    best = max(scores, key=scores.get)
else:
    best = ("ema", True)   # no hold-out: EMA + TTA
print("using", best)

net, tta = cands[best]
pred = predict_probs(net, Xte, tta=tta).argmax(1).cpu().numpy()
out_path = "/kaggle/working/submission.csv" if os.path.exists("/kaggle/working") else "submission.csv"
pd.DataFrame({"ID": np.arange(len(pred)), "target": pred}).to_csv(out_path, index=False)
print("saved", out_path, "| prediction counts min/max:", np.bincount(pred).min(), np.bincount(pred).max())
with open(os.path.join(os.path.dirname(out_path) or ".", "run_summary.json"), "w") as fh:
    json.dump({"epochs": epoch, "val_frac": VAL_FRAC, "used": list(best),
               "val_scores": {f"{k[0]}_tta{k[1]}": v for k, v in scores.items()} if len(Xva) else {},
               "history": history}, fh, indent=1)
print(f"notebook total: {(time.time() - NOTEBOOK_T0) / 60:.1f} min")